In [4]:
from src.state import make_initial_state
from src.schemas import (
    ClarityOutput,
    ResearchItem,
    ResearchOutput,
    ValidationOutput,
    SynthesisOutput,
)

state = make_initial_state("Tell me about Apple")
print("Initial state:")
print(state)

clarity = ClarityOutput(
    clarity_status="clear",
    active_company="Apple",
    last_topic="overview",
    resolved_query="Tell me about Apple",
    clarification_question=None,
)
print("\nClarityOutput OK:")
print(clarity.model_dump())

research = ResearchOutput(
    company="Apple",
    topic="overview",
    resolved_query="Tell me about Apple",
    findings=[
        ResearchItem(
            category="overview",
            title="Apple company overview",
            summary="Apple is a technology company known for consumer devices and software."
        )
    ],
    gaps=[],
    rule_confidence_score=8.0,
    llm_confidence_score=7.0,
    confidence_gate_passed=True,
)
print("\nResearchOutput OK:")
print(research.model_dump())

validation = ValidationOutput(
    validation_result="sufficient",
    validation_feedback="Research is relevant and complete enough."
)
print("\nValidationOutput OK:")
print(validation.model_dump())

synthesis = SynthesisOutput(
    final_response="Apple is a technology company known for consumer devices and software."
)
print("\nSynthesisOutput OK:")
print(synthesis.model_dump())

Initial state:
{'user_query': 'Tell me about Apple', 'resolved_query': None, 'conversation_history': [], 'active_company': None, 'last_topic': None, 'clarity_status': 'needs_clarification', 'clarification_question': None, 'research_output': None, 'rule_confidence_score': None, 'llm_confidence_score': None, 'confidence_gate_passed': None, 'validation_result': None, 'validation_feedback': None, 'attempt_count': 0, 'final_response': None}

ClarityOutput OK:
{'clarity_status': 'clear', 'active_company': 'Apple', 'last_topic': 'overview', 'resolved_query': 'Tell me about Apple', 'clarification_question': None}

ResearchOutput OK:
{'company': 'Apple', 'topic': 'overview', 'resolved_query': 'Tell me about Apple', 'findings': [{'category': 'overview', 'title': 'Apple company overview', 'summary': 'Apple is a technology company known for consumer devices and software.', 'source_hint': None}], 'gaps': [], 'rule_confidence_score': 8.0, 'llm_confidence_score': 7.0, 'confidence_gate_passed': True}


In [8]:
from src.clarity import ClarityAgent
from src.state import make_initial_state

agent = ClarityAgent(
    known_companies=[
        "Apple",
        "Tesla",
        "Microsoft",
        "NVIDIA",
        "Meta",
        "Amazon",
        "Google",
        "OpenAI",
    ]
)

tests = [
    make_initial_state("Tell me about Apple"),
    make_initial_state("Any recent news about Tesla?"),
    {
        **make_initial_state("What about competitors?"),
        "active_company": "Apple",
        "last_topic": "overview",
    },
    {
        **make_initial_state("Tell me more"),
        "active_company": "Apple",
        "last_topic": "ceo",
    },
    {
        **make_initial_state("How is their stock doing?"),
        "active_company": "Apple",
        "last_topic": "overview",
    },
    make_initial_state("Tell me more"),
    make_initial_state("Compare Apple and Tesla"),
]

for i, state in enumerate(tests, start=1):
    result = agent.run(state)
    print(f"\n--- Test {i} ---")
    print("query:", state["user_query"])
    print(result.model_dump())


--- Test 1 ---
query: Tell me about Apple
{'clarity_status': 'clear', 'active_company': 'Apple', 'last_topic': 'overview', 'resolved_query': 'Tell me about Apple', 'clarification_question': None}

--- Test 2 ---
query: Any recent news about Tesla?
{'clarity_status': 'needs_clarification', 'active_company': None, 'last_topic': None, 'resolved_query': None, 'clarification_question': 'I found multiple companies in your request. Please name one company for this version.'}

--- Test 3 ---
query: What about competitors?
{'clarity_status': 'clear', 'active_company': 'What About', 'last_topic': 'competitors', 'resolved_query': 'Tell me about What About competitors', 'clarification_question': None}

--- Test 4 ---
query: Tell me more
{'clarity_status': 'clear', 'active_company': 'Apple', 'last_topic': 'ceo', 'resolved_query': 'Tell me about the CEO of Apple', 'clarification_question': None}

--- Test 5 ---
query: How is their stock doing?
{'clarity_status': 'clear', 'active_company': 'How Is T

In [11]:
from src.research import ResearchAgent
from src.state import make_initial_state

research_agent = ResearchAgent()

state = {
    **make_initial_state("Tell me about Apple"),
    "active_company": "Apple",
    "last_topic": "overview",
    "resolved_query": "Tell me about Apple",
}

result = research_agent.run(state)

print(result.model_dump())

{'company': 'Apple', 'topic': 'overview', 'resolved_query': 'Tell me about Apple', 'findings': [{'category': 'overview', 'title': 'Apple overview', 'summary': 'Apple is a global technology company known for iPhone, Mac, iPad, services, and consumer software.', 'source_hint': 'mock://apple/overview/1'}], 'gaps': [], 'rule_confidence_score': 8.0, 'llm_confidence_score': 10.0, 'confidence_gate_passed': True}


In [12]:
test_states = [
    {
        **make_initial_state("Tell me about Apple"),
        "active_company": "Apple",
        "last_topic": "overview",
        "resolved_query": "Tell me about Apple",
    },
    {
        **make_initial_state("Tell me about the CEO of Apple"),
        "active_company": "Apple",
        "last_topic": "ceo",
        "resolved_query": "Tell me about the CEO of Apple",
    },
    {
        **make_initial_state("Tell me about Tesla financials"),
        "active_company": "Tesla",
        "last_topic": "financials",
        "resolved_query": "Tell me about Tesla financials",
    },
    {
        **make_initial_state("Tell me about Tesla developments"),
        "active_company": "Tesla",
        "last_topic": "developments",
        "resolved_query": "Tell me about recent developments at Tesla",
    },
]

for i, state in enumerate(test_states, start=1):
    result = research_agent.run(state)
    print(f"\n--- Research Test {i} ---")
    print("query:", state["resolved_query"])
    print("topic:", result.topic)
    print("findings:", len(result.findings))
    print("gaps:", result.gaps)
    print("rule_confidence_score:", result.rule_confidence_score)
    print("llm_confidence_score:", result.llm_confidence_score)
    print("confidence_gate_passed:", result.confidence_gate_passed)


--- Research Test 1 ---
query: Tell me about Apple
topic: overview
findings: 1
gaps: []
rule_confidence_score: 8.0
llm_confidence_score: 10.0
confidence_gate_passed: True

--- Research Test 2 ---
query: Tell me about the CEO of Apple
topic: ceo
findings: 1
gaps: []
rule_confidence_score: 8.0
llm_confidence_score: 10.0
confidence_gate_passed: True

--- Research Test 3 ---
query: Tell me about Tesla financials
topic: financials
findings: 0
gaps: ["No research findings retrieved for topic 'financials'."]
rule_confidence_score: 1.0
llm_confidence_score: 1.0
confidence_gate_passed: False

--- Research Test 4 ---
query: Tell me about recent developments at Tesla
topic: developments
findings: 0
gaps: ["No research findings retrieved for topic 'developments'."]
rule_confidence_score: 1.0
llm_confidence_score: 1.0
confidence_gate_passed: False


In [13]:
retry_state = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
    "attempt_count": 2,  # this makes ResearchAgent run attempt 3
    "validation_feedback": "Missing financials. Try broader overview and any available news or financials evidence.",
}

retry_result = research_agent.run(retry_state)
print(retry_result.model_dump())

{'company': 'Tesla', 'topic': 'financials', 'resolved_query': 'Tell me about Tesla financials', 'findings': [{'category': 'overview', 'title': 'Tesla overview', 'summary': 'Tesla is an electric vehicle and energy company focused on EVs, batteries, and related technology.', 'source_hint': 'mock://tesla/overview/1'}, {'category': 'news', 'title': 'Tesla recent news', 'summary': 'Tesla remains a major focus in EV market coverage, production discussion, and product rollout commentary.', 'source_hint': 'mock://tesla/news/1'}], 'gaps': ["Missing topic-specific evidence for 'financials'."], 'rule_confidence_score': 5.0, 'llm_confidence_score': 3.0, 'confidence_gate_passed': False}


In [14]:
from src.research import ResearchAgent
from src.validator import ValidatorAgent
from src.state import make_initial_state

research_agent = ResearchAgent()
validator_agent = ValidatorAgent()

state = {
    **make_initial_state("Tell me about Apple"),
    "active_company": "Apple",
    "last_topic": "overview",
    "resolved_query": "Tell me about Apple",
}

research_result = research_agent.run(state)
state["research_output"] = research_result.model_dump()

validation_result = validator_agent.run(state)

print("Research:")
print(research_result.model_dump())
print("\nValidation:")
print(validation_result.model_dump())

Research:
{'company': 'Apple', 'topic': 'overview', 'resolved_query': 'Tell me about Apple', 'findings': [{'category': 'overview', 'title': 'Apple overview', 'summary': 'Apple is a global technology company known for iPhone, Mac, iPad, services, and consumer software.', 'source_hint': 'mock://apple/overview/1'}], 'gaps': [], 'rule_confidence_score': 8.0, 'llm_confidence_score': 10.0, 'confidence_gate_passed': True}

Validation:
{'validation_result': 'sufficient', 'validation_feedback': "Research is sufficient for 'Tell me about Apple'."}


In [15]:
state = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
}

research_result = research_agent.run(state)
state["research_output"] = research_result.model_dump()

validation_result = validator_agent.run(state)

print("Research:")
print(research_result.model_dump())
print("\nValidation:")
print(validation_result.model_dump())

Research:
{'company': 'Tesla', 'topic': 'financials', 'resolved_query': 'Tell me about Tesla financials', 'findings': [], 'gaps': ["No research findings retrieved for topic 'financials'."], 'rule_confidence_score': 1.0, 'llm_confidence_score': 1.0, 'confidence_gate_passed': False}

Validation:
{'validation_result': 'insufficient', 'validation_feedback': "No findings were retrieved for topic 'financials'. Try a direct financials lookup for the company."}


In [16]:
retry_state = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
    "attempt_count": 2,
    "validation_feedback": "Need topic-specific evidence for financials. Try broader financial or stock-related evidence, then fall back to overview only as supporting context.",
}

retry_research = research_agent.run(retry_state)
retry_state["research_output"] = retry_research.model_dump()

retry_validation = validator_agent.run(retry_state)

print("Retry Research:")
print(retry_research.model_dump())
print("\nRetry Validation:")
print(retry_validation.model_dump())

Retry Research:
{'company': 'Tesla', 'topic': 'financials', 'resolved_query': 'Tell me about Tesla financials', 'findings': [{'category': 'overview', 'title': 'Tesla overview', 'summary': 'Tesla is an electric vehicle and energy company focused on EVs, batteries, and related technology.', 'source_hint': 'mock://tesla/overview/1'}], 'gaps': ["Missing topic-specific evidence for 'financials'."], 'rule_confidence_score': 4.0, 'llm_confidence_score': 3.0, 'confidence_gate_passed': False}

Retry Validation:
{'validation_result': 'insufficient', 'validation_feedback': "Current findings are not relevant enough to answer 'Tell me about Tesla financials'. Need topic-specific evidence for 'financials'. Research gaps detected: Missing topic-specific evidence for 'financials'.. Do not answer from unrelated categories. Retrieve direct evidence for the requested topic only. Try broader financial or stock-related evidence, then fall back to overview only as supporting context."}


In [17]:
# sufficient case
state_ok = {
    **make_initial_state("Tell me about Apple"),
    "active_company": "Apple",
    "last_topic": "overview",
    "resolved_query": "Tell me about Apple",
}
r_ok = research_agent.run(state_ok)
state_ok["research_output"] = r_ok.model_dump()
v_ok = validator_agent.run(state_ok)

assert v_ok.validation_result == "sufficient"

# insufficient case
state_bad = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
}
r_bad = research_agent.run(state_bad)
state_bad["research_output"] = r_bad.model_dump()
v_bad = validator_agent.run(state_bad)

assert v_bad.validation_result == "insufficient"
assert "financial" in v_bad.validation_feedback.lower()

print("Validator assertions passed.")

Validator assertions passed.


In [18]:
from src.research import ResearchAgent
from src.validator import ValidatorAgent
from src.synthesis import SynthesisAgent
from src.state import make_initial_state

research_agent = ResearchAgent()
validator_agent = ValidatorAgent()
synthesis_agent = SynthesisAgent()

state = {
    **make_initial_state("Tell me about Apple"),
    "active_company": "Apple",
    "last_topic": "overview",
    "resolved_query": "Tell me about Apple",
}

research_result = research_agent.run(state)
state["research_output"] = research_result.model_dump()

validation_result = validator_agent.run(state)
state["validation_result"] = validation_result.validation_result
state["validation_feedback"] = validation_result.validation_feedback

synthesis_result = synthesis_agent.run(state)

print(synthesis_result.model_dump()["final_response"])

Here is a grounded summary of Apple's overview:
- Apple overview: Apple is a global technology company known for iPhone, Mac, iPad, services, and consumer software.


In [20]:
state = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
    "attempt_count": 3,
}

research_result = research_agent.run(state)
state["research_output"] = research_result.model_dump()

validation_result = validator_agent.run(state)
state["validation_result"] = validation_result.validation_result
state["validation_feedback"] = validation_result.validation_feedback

synthesis_result = synthesis_agent.run(state)

print(synthesis_result.model_dump()["final_response"])

I could not gather enough complete information to fully answer this request.
Here is the best available summary based on the retrieved research.

I do not have enough topic-specific retrieved information for Tesla's financials.

Supporting context from other retrieved categories:
- (overview) Tesla is an electric vehicle and energy company focused on EVs, batteries, and related technology.

Known limitation: Current findings are not relevant enough to answer 'Tell me about Tesla financials'. Need topic-specific evidence for 'financials'. Research gaps detected: Missing topic-specific evidence for 'financials'.. Do not answer from unrelated categories. Retrieve direct evidence for the requested topic only. Try broader financial or stock-related evidence, then fall back to overview only as supporting context.


In [21]:
state = {
    **make_initial_state("Tell me about Tesla CEO"),
    "active_company": "Tesla",
    "last_topic": "ceo",
    "resolved_query": "Tell me about the CEO of Tesla",
    "attempt_count": 1,
}

research_result = research_agent.run(state)
state["research_output"] = research_result.model_dump()

validation_result = validator_agent.run(state)
state["validation_result"] = validation_result.validation_result
state["validation_feedback"] = validation_result.validation_feedback

synthesis_result = synthesis_agent.run(state)

print("Validation:", validation_result.model_dump())
print()
print(synthesis_result.model_dump()["final_response"])

Validation: {'validation_result': 'sufficient', 'validation_feedback': "Research is sufficient for 'Tell me about the CEO of Tesla'."}

Here is a grounded summary of Tesla's CEO / leadership:
- Tesla CEO: Elon Musk is the CEO of Tesla.


In [22]:
# sufficient summary
state_ok = {
    **make_initial_state("Tell me about Apple"),
    "active_company": "Apple",
    "last_topic": "overview",
    "resolved_query": "Tell me about Apple",
}
r_ok = research_agent.run(state_ok)
state_ok["research_output"] = r_ok.model_dump()
v_ok = validator_agent.run(state_ok)
state_ok["validation_result"] = v_ok.validation_result
state_ok["validation_feedback"] = v_ok.validation_feedback
s_ok = synthesis_agent.run(state_ok)

assert "Apple" in s_ok.final_response

# incomplete summary
state_bad = {
    **make_initial_state("Tell me about Tesla financials"),
    "active_company": "Tesla",
    "last_topic": "financials",
    "resolved_query": "Tell me about Tesla financials",
    "attempt_count": 3,
}
r_bad = research_agent.run(state_bad)
state_bad["research_output"] = r_bad.model_dump()
v_bad = validator_agent.run(state_bad)
state_bad["validation_result"] = v_bad.validation_result
state_bad["validation_feedback"] = v_bad.validation_feedback
s_bad = synthesis_agent.run(state_bad)

assert "could not gather enough complete information" in s_bad.final_response.lower()
assert "financial" in s_bad.final_response.lower()

print("Synthesis assertions passed.")

Synthesis assertions passed.
